# 03. Sistema Especialista Baseado em Regras SE–ENTÃO

**DCT1401 — Inteligência Artificial (UFRN, 2026.2)**  
**Aluna:** Mariana dos Santos Batista Medeiros (Matrícula: 20230083988)

**Atividade 02 — Avaliação Comparativa de Modelos**

Este notebook desenvolve e avalia um Sistema Especialista baseado em regras SE–ENTÃO para classificar a variável `Cardiovascular_Disease`, conforme os requisitos da Seção 4 do enunciado.

A construção contempla a análise de associações observadas nos dados de treinamento, a definição de 11 regras, sua classificação tipológica, a organização da base de conhecimento e a implementação do motor de inferência.

As regras são de natureza acadêmica e exploratória, não representando critérios de diagnóstico clínico.

São utilizadas as mesmas seis variáveis preditoras selecionadas na Atividade 01, com divisão de 80% para treinamento e 20% para teste, estratificação da classificação e `random_state=42`.

O pré-processamento e a obtenção dos escores empíricos são realizados exclusivamente com os dados de treinamento, evitando *data leakage*. A base de regras permanece fixa durante a avaliação final no conjunto de teste.

O desempenho será analisado por meio de métricas de classificação, considerando também a interpretabilidade e as limitações dessa abordagem.


**Configuração do ambiente — Google Colab**

Esta etapa conecta o Google Colab ao Google Drive e configura os caminhos necessários para acessar o dataset original e os módulos Python do projeto.

A execução pressupõe que a pasta `projeto-cardiovascular-ia-02` esteja localizada em `Meu Drive`, conforme as instruções disponíveis no `README.md` da Atividade 02.

O código verifica a existência do dataset e prepara o ambiente para a execução das etapas seguintes.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import os
import sys

# 1. Conectar o Google Drive ao Colab
drive.mount("/content/drive")

# 2. Identificar a pasta principal do projeto
PROJETO = Path(
    "/content/drive/MyDrive/projeto-cardiovascular-ia-02"
)

# 3. Verificar se o dataset original está disponível
ARQUIVO_RAW = (
    PROJETO / "data/raw/cardiovascular_disease_prediction.csv"
)

if not ARQUIVO_RAW.is_file():
    raise FileNotFoundError(
        f"Dataset não encontrado: {ARQUIVO_RAW}"
    )

# 4. Definir a pasta notebooks como diretório de execução
os.chdir(PROJETO / "notebooks")

# 5. Permitir a importação dos módulos da pasta src
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

print("Projeto localizado com sucesso!")
print("Diretório de execução:", Path.cwd())
print("Dataset original encontrado:", ARQUIVO_RAW.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projeto localizado com sucesso!
Diretório de execução: /content/drive/MyDrive/projeto-cardiovascular-ia-02/notebooks
Dataset original encontrado: cardiovascular_disease_prediction.csv




## 1. Preparação dos dados e protocolo experimental

Esta seção divide o dataset bruto em treinamento (80%) e teste (20%), utilizando estratificação pela variável `Cardiovascular_Disease`.

**Justificativa do protocolo.** Embora o enunciado sugira reutilizar os arquivos `data/processed/dataset_classification.csv` e `dataset_regression.csv`, esses conjuntos foram preparados anteriormente com transformações ajustadas sobre os dados completos, antes da divisão entre treinamento e teste. Sua reutilização poderia introduzir *data leakage*.

Por isso, optou-se por partir do CSV original, preservando as seis variáveis selecionadas na Atividade 01 e realizando a divisão antes do pré-processamento. As estatísticas necessárias à imputação e à codificação são aprendidas exclusivamente no treinamento e posteriormente aplicadas ao teste.

In [ ]:
import importlib
import src.pipeline_data as pipeline_data

# Recarrega o módulo, caso a versão antiga esteja em memória
importlib.reload(pipeline_data)

# Confere se estamos utilizando a versão corrigida
print("Features selecionadas:")
print(pipeline_data.FEATURES_MODELO)

# Divisão a partir do CSV original
(
    X_train,
    X_test,
    y_train_c,
    y_test_c,
    y_train_r,
    y_test_r,
    feat_cols
) = pipeline_data.carregar_e_dividir_dados_sem_leakage(
    "../data/raw/cardiovascular_disease_prediction.csv"
)

print("\nDimensão do treinamento:", X_train.shape)
print("Dimensão do teste:", X_test.shape)

print("\nProporção de positivos no treinamento:", y_train_c.mean())

print("\nValores ausentes:")
print("Treinamento:", X_train.isna().sum().sum())
print("Teste:", X_test.isna().sum().sum())

Features selecionadas:
('Diabetes', 'Hypertension', 'Chest_Pain', 'ECG_Result', 'Blood_Glucose', 'Blood_Pressure_Systolic')

Dimensão do treinamento: (16000, 6)
Dimensão do teste: (4000, 6)

Proporção de positivos no treinamento: 0.306

Valores ausentes:
Treinamento: 476
Teste: 124


In [ ]:
# Preparação dos dados para o Sistema Especialista
X_train_se, X_test_se, preprocessador_se = (
    pipeline_data.preparar_dados_sistema_especialista(
        X_train, X_test
    )
)

print("Dimensão do treinamento:", X_train_se.shape)
print("Dimensão do teste:", X_test_se.shape)

print("\nValores ausentes após o pré-processamento:")
print("Treinamento:", X_train_se.isna().sum().sum())
print("Teste:", X_test_se.isna().sum().sum())

print("\nMediana da glicose no treinamento original:")
print(X_train["Blood_Glucose"].median())

print("\nPrimeiros registros transformados:")
display(X_train_se.head())

Dimensão do treinamento: (16000, 6)
Dimensão do teste: (4000, 6)

Valores ausentes após o pré-processamento:
Treinamento: 0
Teste: 0

Mediana da glicose no treinamento original:
105.0

Primeiros registros transformados:


,Diabetes,Hypertension,Chest_Pain,ECG_Result,Blood_Glucose,Blood_Pressure_Systolic
10656,0.0,0.0,0.0,0.0,87.0,127.0
16130,0.0,0.0,0.0,0.0,92.0,117.0
6870,0.0,0.0,0.0,1.0,118.0,102.0
4331,0.0,0.0,0.0,0.0,124.0,133.0
3982,0.0,0.0,1.0,0.0,120.0,139.0


## 2. Evidências empíricas para as regras

As análises desta seção usam **somente o conjunto de treinamento**. Frequências por perfil são evidências descritivas, não probabilidades clínicas nem estimativas de desempenho fora da amostra.


In [ ]:
# Análise das combinações das variáveis binárias
# Utiliza exclusivamente o conjunto de treinamento

variaveis_binarias = [
    "Diabetes",
    "Hypertension",
    "Chest_Pain",
    "ECG_Result"
]

# Reunir as features e o target do treinamento
analise_regras = X_train_se.copy()

analise_regras["Cardiovascular_Disease"] = y_train_c

# Calcular quantidade de registros e proporção da classe 1
tabela_combinacoes = (
    analise_regras
    .groupby(variaveis_binarias)["Cardiovascular_Disease"]
    .agg(
        quantidade="count",
        casos_positivos="sum",
        proporcao_positivos="mean"
    )
    .reset_index()
    .sort_values(
        "proporcao_positivos",
        ascending=False
    )
)

# Exibir proporções como percentuais
tabela_combinacoes["percentual_positivos"] = (
    tabela_combinacoes["proporcao_positivos"] * 100
).round(2)

display(
    tabela_combinacoes[
        variaveis_binarias
        + ["quantidade", "casos_positivos", "percentual_positivos"]
    ]
)

,Diabetes,Hypertension,Chest_Pain,ECG_Result,quantidade,casos_positivos,percentual_positivos
15,1.0,1.0,1.0,1.0,71,69,97.18
14,1.0,1.0,1.0,0.0,234,215,91.88
13,1.0,1.0,0.0,1.0,274,246,89.78
11,1.0,0.0,1.0,1.0,183,164,89.62
7,0.0,1.0,1.0,1.0,184,157,85.33
12,1.0,1.0,0.0,0.0,808,522,64.60
10,1.0,0.0,1.0,0.0,563,344,61.10
9,1.0,0.0,0.0,1.0,716,419,58.52
5,0.0,1.0,0.0,1.0,677,386,57.02
6,0.0,1.0,1.0,0.0,532,300,56.39


### 2.1 Variáveis numéricas e limites exploratórios

As distribuições de `Blood_Glucose` e `Blood_Pressure_Systolic` são estudadas **apenas no treinamento**. Na análise por quartis individuais, utilizam-se os valores originalmente observados, excluindo os ausentes; na análise conjunta, utilizam-se os dados preparados, com imputação aprendida somente no treino. Os pontos de corte são **exploratórios**, sem significado de diagnóstico clínico.

In [ ]:
# Analisar as duas features numéricas por classe
# Utilizar exclusivamente os dados de treinamento

variaveis_numericas = [
    "Blood_Glucose",
    "Blood_Pressure_Systolic"
]

resumo_numerico = (
    analise_regras
    .groupby("Cardiovascular_Disease")[variaveis_numericas]
    .agg(["count", "mean", "median", "min", "max"])
)

display(resumo_numerico.round(2))

Blood_Glucose                              \
                               count    mean median   min    max   
Cardiovascular_Disease                                             
0                              11104  101.49  102.0  60.0  207.0   
1                               4896  114.84  118.0  60.0  215.0   

                       Blood_Pressure_Systolic                              
                                         count    mean median   min    max  
Cardiovascular_Disease                                                      
0                                        11104  122.38  122.0  90.0  185.0  
1                                         4896  130.36  131.0  90.0  193.0

In [ ]:
# Análise por quartis das features numéricas
# Apenas dados de treinamento, sem utilizar o teste

variaveis_numericas = [
    "Blood_Glucose",
    "Blood_Pressure_Systolic"
]

for variavel in variaveis_numericas:

    # Utilizar valores originais do treinamento,
    # antes da imputação
    dados = pd.DataFrame({
        "valor": X_train[variavel],
        "classe": y_train_c
    }).dropna()

    # Dividir a variável em quatro faixas
    dados["faixa"] = pd.qcut(
        dados["valor"],
        q=4,
        duplicates="drop"
    )

    # Calcular a frequência da classe positiva
    resultado = (
        dados.groupby("faixa", observed=True)
        .agg(
            quantidade=("classe", "count"),
            positivos=("classe", "sum"),
            percentual_positivos=("classe", "mean")
        )
        .reset_index()
    )

    resultado["percentual_positivos"] = (
        resultado["percentual_positivos"] * 100
    ).round(2)

    print(f"\nAnálise por faixas: {variavel}")
    display(resultado)


Análise por faixas: Blood_Glucose


,faixa,quantidade,positivos,percentual_positivos
0,"(59.999, 84.0]",3900,900,23.08
1,"(84.0, 105.0]",4004,894,22.33
2,"(105.0, 125.0]",3824,890,23.27
3,"(125.0, 215.0]",3796,2078,54.74



Análise por faixas: Blood_Pressure_Systolic


,faixa,quantidade,positivos,percentual_positivos
0,"(89.999, 113.0]",4256,967,22.72
1,"(113.0, 125.0]",4077,1006,24.68
2,"(125.0, 137.0]",3883,959,24.70
3,"(137.0, 193.0]",3784,1964,51.90


In [ ]:
# Análise conjunta da Blood_Glucose e da Blood_Pressure_Systolic
# Exclusivamente no conjunto de treinamento

analise_numerica = X_train_se[
    ["Blood_Glucose", "Blood_Pressure_Systolic"]
].copy()

analise_numerica["Cardiovascular_Disease"] = y_train_c

# Limites exploratórios obtidos dos quartis do treinamento
analise_numerica["Glicose_acima_125"] = (
    analise_numerica["Blood_Glucose"] > 125
)

analise_numerica["Pressao_acima_137"] = (
    analise_numerica["Blood_Pressure_Systolic"] > 137
)

resultado_conjunto = (
    analise_numerica
    .groupby(["Glicose_acima_125", "Pressao_acima_137"])[
        "Cardiovascular_Disease"
    ]
    .agg(
        quantidade="count",
        positivos="sum",
        proporcao="mean"
    )
    .reset_index()
)

resultado_conjunto["percentual_positivos"] = (
    resultado_conjunto["proporcao"] * 100
).round(2)

display(
    resultado_conjunto[
        [
            "Glicose_acima_125",
            "Pressao_acima_137",
            "quantidade",
            "positivos",
            "percentual_positivos"
        ]
    ]
)

,Glicose_acima_125,Pressao_acima_137,quantidade,positivos,percentual_positivos
0,False,False,9281,1548,16.68
1,False,True,2923,1270,43.45
2,True,False,2935,1384,47.16
3,True,True,861,694,80.60


### 2.2 Análise das regras candidatas, cobertura e conflitos

Com base nas análises realizadas no conjunto de treinamento, foram definidas inicialmente cinco regras candidatas para classificação. Nesta etapa, serão avaliadas sua cobertura e a ocorrência de conflitos entre as conclusões, a fim de identificar possíveis limitações e orientar o refinamento da base de conhecimento.

In [ ]:
import operator

# Primeiras regras candidatas, obtidas das análises do treino
regras_candidatas = [
    {
        "id": "RC01",
        "tipo": "Relação",
        "condicoes": [
            ("Diabetes", "==", 1),
            ("Hypertension", "==", 1),
            ("Chest_Pain", "==", 1),
            ("ECG_Result", "==", 1)
        ],
        "conclusao": 1
    },
    {
        "id": "RC02",
        "tipo": "Relação",
        "condicoes": [
            ("Diabetes", "==", 1),
            ("Hypertension", "==", 1),
            ("Chest_Pain", "==", 1)
        ],
        "conclusao": 1
    },
    {
        "id": "RC03",
        "tipo": "Heurística",
        "condicoes": [
            ("Blood_Glucose", ">", 125),
            ("Blood_Pressure_Systolic", ">", 137)
        ],
        "conclusao": 1
    },
    {
        "id": "RC04",
        "tipo": "Relação",
        "condicoes": [
            ("Diabetes", "==", 0),
            ("Hypertension", "==", 0),
            ("Chest_Pain", "==", 0),
            ("ECG_Result", "==", 0)
        ],
        "conclusao": 0
    },
    {
        "id": "RC05",
        "tipo": "Heurística",
        "condicoes": [
            ("Blood_Glucose", "<=", 125),
            ("Blood_Pressure_Systolic", "<=", 137)
        ],
        "conclusao": 0
    }
]

operadores = {
    "==": operator.eq,
    ">": operator.gt,
    "<=": operator.le
}

def verificar_condicoes(dados, condicoes):
    mascara = pd.Series(True, index=dados.index)

    for feature, operador, valor in condicoes:
        mascara &= operadores[operador](dados[feature], valor)

    return mascara

resumo_regras = []
ativacoes = pd.DataFrame(index=X_train_se.index)

for regra in regras_candidatas:
    mascara = verificar_condicoes(
        X_train_se, regra["condicoes"]
    )

    ativacoes[regra["id"]] = mascara

    quantidade = int(mascara.sum())
    positivos = int(y_train_c.loc[mascara].sum())

    acertos = (
        positivos if regra["conclusao"] == 1
        else quantidade - positivos
    )

    resumo_regras.append({
        "Regra": regra["id"],
        "Tipo": regra["tipo"],
        "Conclusao": regra["conclusao"],
        "Cobertura": quantidade,
        "Acertos_na_conclusao": acertos,
        "Frequencia_acerto_%": round(
            100 * acertos / quantidade, 2
        )
    })

display(pd.DataFrame(resumo_regras))

# Verificar cobertura e conflitos
alguma_regra = ativacoes.any(axis=1)

ids_positivos = [
    r["id"] for r in regras_candidatas
    if r["conclusao"] == 1
]
ids_negativos = [
    r["id"] for r in regras_candidatas
    if r["conclusao"] == 0
]

conflitos = (
    ativacoes[ids_positivos].any(axis=1)
    & ativacoes[ids_negativos].any(axis=1)
)

print("Registros cobertos:", int(alguma_regra.sum()))
print("Registros sem regra:", int((~alguma_regra).sum()))
print("Registros com conflito:", int(conflitos.sum()))

,Regra,Tipo,Conclusao,Cobertura,Acertos_na_conclusao,Frequencia_acerto_%
0,RC01,Relação,1,71,69,97.18
1,RC02,Relação,1,305,284,93.11
2,RC03,Heurística,1,861,694,80.60
3,RC04,Relação,0,4599,4314,93.80
4,RC05,Heurística,0,9281,7733,83.32


Registros cobertos: 10442
Registros sem regra: 5558
Registros com conflito: 22


In [ ]:
# Diagnóstico da cobertura e dos conflitos
# Exclusivamente com dados de treinamento

sem_regra = ~ativacoes.any(axis=1)

com_conflito = (
    ativacoes[ids_positivos].any(axis=1)
    & ativacoes[ids_negativos].any(axis=1)
)

diagnostico = pd.DataFrame({
    "Situacao": [
        "Sem regra aplicável",
        "Com regras conflitantes"
    ],
    "Quantidade": [
        int(sem_regra.sum()),
        int(com_conflito.sum())
    ],
    "Positivos": [
        int(y_train_c.loc[sem_regra].sum()),
        int(y_train_c.loc[com_conflito].sum())
    ]
})

diagnostico["Percentual_positivos"] = (
    100 * diagnostico["Positivos"]
    / diagnostico["Quantidade"]
).round(2)

display(diagnostico)

# Identificar quais regras entram em conflito
detalhes_conflitos = pd.DataFrame({
    "Regras_ativadas": ativacoes.loc[com_conflito].apply(
        lambda linha: ", ".join(linha.index[linha]),
        axis=1
    ),
    "Classe_real": y_train_c.loc[com_conflito]
})

print("\nCombinações de regras conflitantes:")

display(
    detalhes_conflitos
    .groupby(["Regras_ativadas", "Classe_real"])
    .size()
    .unstack(fill_value=0)
)

,Situacao,Quantidade,Positivos,Percentual_positivos
0,Sem regra aplicável,5558,2532,45.56
1,Com regras conflitantes,22,18,81.82



Combinações de regras conflitantes:


Classe_real,0,1
Regras_ativadas,,
"RC01, RC02, RC05",1,6
"RC02, RC05",3,12


### 2.3 Refinamento e consolidação da base de regras

A análise das cinco regras iniciais identificou **5.558 registros sem cobertura e 22 casos de conflito** entre conclusões. Diante dessas limitações, a base foi ampliada para 11 regras, considerando novas associações identificadas exclusivamente no conjunto de treinamento.

Com o refinamento, a cobertura passou de **10.442 para 12.985 registros (81,16%)**. Os 3.015 registros restantes são tratados pelo mecanismo de resposta padrão (*fallback*).

Para resolver possíveis conflitos, foi definida uma ordem de prioridade, na qual prevalece a primeira regra cujas condições são satisfeitas.

A base definitiva, contendo as 11 regras SE–ENTÃO, está documentada no arquivo `rules/regras_classificacao.txt`.

## 3. Base de conhecimento e motor de inferência

A base de conhecimento está armazenada no arquivo `rules/regras_classificacao.txt`, separada do motor de inferência implementado em `src/expert_system.py`.

O motor utiliza uma **forma simplificada de encadeamento para frente**, partindo dos fatos disponíveis em cada registro (as seis variáveis) para verificar as regras SE–ENTÃO.

Como todas as regras concluem diretamente a classificação `Cardiovascular_Disease`, não há geração de fatos intermediários nem necessidade de múltiplos ciclos de inferência. Quando mais de uma regra é satisfeita, prevalece a primeira na ordem de prioridade estabelecida no arquivo TXT.

As frequências utilizadas como escores são calculadas exclusivamente no conjunto de treinamento.

In [ ]:
import importlib
import src.expert_system as modulo_se

# Atualizar o módulo carregado no Colab
importlib.reload(modulo_se)

from src.expert_system import (
    BaseConhecimentoClassificacao,
    MotorInferenciaClassificacao
)

# Carregar a base de conhecimento externa
base = BaseConhecimentoClassificacao(
    "../rules/regras_classificacao.txt"
)

# Calcular evidências somente com o treinamento
estatisticas = base.analisar_no_treinamento(
    X_train_se,
    y_train_c
)

# Construir o motor de inferência
motor = MotorInferenciaClassificacao(base)

print("Regras carregadas:", len(base.regras))
print("Registros sem regra:", base.quantidade_sem_regra)
print("Classe padrão:", base.classe_padrao)

display(
    estatisticas[
        [
            "Regra",
            "Tipo",
            "Classe",
            "Cobertura_bruta",
            "Cobertura_efetiva",
            "Frequencia_acerto_efetivo"
        ]
    ].round(4)
)

Regras carregadas: 11
Registros sem regra: 3015
Classe padrão: 0


,Regra,Tipo,Classe,Cobertura_bruta,Cobertura_efetiva,Frequencia_acerto_efetivo
0,RC01,Relação,1,71,71,0.9718
1,RC02,Relação,1,305,234,0.9188
2,RC04,Relação,0,4599,4599,0.9380
3,RC03,Heurística,1,861,699,0.7682
4,RC06,Heurística,1,1387,502,0.5857
5,RC07,Heurística,1,1051,721,0.6782
6,RC08,Heurística,1,864,610,0.6492
7,RC09,Heurística,1,1021,532,0.5639
8,RC10,Heurística,1,1244,696,0.5819
9,RC11,Heurística,1,1206,677,0.5702


### 3.1 Rastreabilidade e verificação no treinamento

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# Inferência exclusivamente no treinamento
previsoes, escores, regras_utilizadas = motor.inferir(
    X_train_se
)

# Verificações de integridade
assert len(previsoes) == len(X_train_se)
assert len(regras_utilizadas) == len(X_train_se)

assert np.array_equal(
    previsoes,
    motor.predict(X_train_se)
)

assert np.all((escores >= 0) & (escores <= 1))

# Rastreabilidade das decisões
rastreabilidade = X_train_se.copy()

rastreabilidade["Classe_real"] = y_train_c
rastreabilidade["Classe_prevista"] = previsoes
rastreabilidade["Regra_aplicada"] = regras_utilizadas
rastreabilidade["Escore_positivo"] = escores

print("Exemplos de classificações realizadas:")
display(rastreabilidade.head(10))

print("\nMatriz de confusão — treinamento:")
print(confusion_matrix(y_train_c, previsoes, labels=[0, 1]))

print("\nRelatório de classificação — treinamento:")
print(
    classification_report(
        y_train_c,
        previsoes,
        digits=4,
        zero_division=0
    )
)

# Conferir as decisões produzidas pelo fallback
mask_fallback = regras_utilizadas == "FALLBACK"

print("\nRegistros classificados pelo fallback:",
      int(mask_fallback.sum()))

print("Classe real dos registros de fallback:")
print(
    y_train_c.loc[
        X_train_se.index[mask_fallback]
    ].value_counts().sort_index()
)

Exemplos de classificações realizadas:


,Diabetes,Hypertension,Chest_Pain,ECG_Result,Blood_Glucose,Blood_Pressure_Systolic,Classe_real,Classe_prevista,Regra_aplicada,Escore_positivo
10656,0.0,0.0,0.0,0.0,87.0,127.0,0,0,RC04,0.061970
16130,0.0,0.0,0.0,0.0,92.0,117.0,0,0,RC04,0.061970
6870,0.0,0.0,0.0,1.0,118.0,102.0,0,0,RC05,0.184138
4331,0.0,0.0,0.0,0.0,124.0,133.0,0,0,RC04,0.061970
3982,0.0,0.0,1.0,0.0,120.0,139.0,0,0,FALLBACK,0.281592
19650,1.0,0.0,0.0,0.0,139.0,128.0,1,0,FALLBACK,0.281592
11021,0.0,0.0,0.0,0.0,97.0,133.0,0,0,RC04,0.061970
13192,0.0,1.0,0.0,0.0,101.0,145.0,0,0,FALLBACK,0.281592
5051,0.0,0.0,0.0,1.0,112.0,109.0,0,0,RC05,0.184138
2016,0.0,1.0,0.0,0.0,65.0,139.0,1,0,FALLBACK,0.281592



Matriz de confusão — treinamento:
[[9453 1651]
 [1805 3091]]

Relatório de classificação — treinamento:
              precision    recall  f1-score   support

           0     0.8397    0.8513    0.8455     11104
           1     0.6518    0.6313    0.6414      4896

    accuracy                         0.7840     16000
   macro avg     0.7458    0.7413    0.7434     16000
weighted avg     0.7822    0.7840    0.7830     16000


Registros classificados pelo fallback: 3015
Classe real dos registros de fallback:
Cardiovascular_Disease
0    2166
1     849
Name: count, dtype: int64


## 4. Registro do esforço humano

As três sessões de elaboração e revisão da base de regras estão registradas em `logs/cronometro_regras.csv`. O notebook **apenas lê** esse arquivo e calcula os indicadores, sem acrescentar sessões automaticamente. O registro representa o trabalho humano informado, distinto do tempo de geração automatizada e do tempo de inferência. A quantidade anotada em `regras_criadas` corresponde às regras efetivamente elaboradas ou reformuladas em cada sessão.


In [ ]:
import pandas as pd

arquivo = PROJETO / "logs/cronometro_regras.csv"

df_tempo = pd.read_csv(arquivo)

# Conferir a consistência dos tempos registrados
diferenca = (
    df_tempo["tempo_relogio_min"]
    - df_tempo["tempo_interrupcoes_min"]
    - df_tempo["tempo_minutos"]
)

assert (diferenca.abs() < 0.000001).all(), (
    "Inconsistência entre duração, pausas e tempo efetivo."
)

print(
    "Tempo total decorrido:",
    df_tempo["tempo_relogio_min"].sum(),
    "min"
)

print(
    "Tempo total de pausas:",
    df_tempo["tempo_interrupcoes_min"].sum(),
    "min"
)

# Conferência das sessões
display(df_tempo)

# Cálculo das estatísticas
total = df_tempo["tempo_minutos"].sum()
media = df_tempo["tempo_minutos"].mean()
desvio = df_tempo["tempo_minutos"].std(ddof=1)
qtd_regras = df_tempo["regras_criadas"].sum()
media_regra = total / qtd_regras

print(f"Sessões registradas: {len(df_tempo)}")
print(f"Tempo total: {total:.2f} minutos")
print(f"Tempo médio por sessão: {media:.2f} minutos")
print(f"Desvio padrão amostral: {desvio:.2f} minutos")
print(f"Regras registradas: {qtd_regras}")
print(f"Tempo médio por regra: {media_regra:.2f} minutos")

Tempo total decorrido: 75 min
Tempo total de pausas: 15.0 min


,sessao,tarefa,data,horario_inicio,horario_fim,tempo_relogio_min,tempo_interrupcoes_min,tempo_minutos,regras_criadas,interrupcoes,descricao
0,1,Classificacao,2026-10-16,18:00,18:30,30,10.0,20.0,5,Pausa,Criação de regras
1,2,Classificacao,2026-10-16,18:50,19:15,25,3.0,22.0,4,Descanso,Elaboração e pesquisa
2,3,Classificacao,2026-10-16,19:20,19:40,20,2.0,18.0,2,Pausa,Pesquisa e elaboração de regras


Sessões registradas: 3
Tempo total: 60.00 minutos
Tempo médio por sessão: 20.00 minutos
Desvio padrão amostral: 2.00 minutos
Regras registradas: 11
Tempo médio por regra: 5.45 minutos


## 5. Eficiência computacional da inferência

Os experimentos são executados no ambiente computacional disponibilizado pelo **Google Colab**, cujas características de processador, memória e versões de bibliotecas são registradas para documentar as condições de execução.

A inferência é medida em 30 repetições sobre 4.000 registros do treinamento. O `tracemalloc` acompanha alocações de memória gerenciadas pelo Python, não o consumo total de RAM.

O consumo de energia é **estimado**, utilizando a fórmula potência × tempo de inferência. Foi adotada uma potência hipotética de 45 W, e não uma medição elétrica real. As emissões são estimadas com o fator de 45,8 gCO₂/kWh indicado na atividade. O mesmo procedimento será utilizado na comparação com árvores de decisão e ensembles.

In [ ]:
import os
import platform
import sys
import sklearn
import numpy
import pandas

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
except ImportError:
    ram_gb = float("nan")

cpu = platform.processor() or "Não identificado"

try:
    with open("/proc/cpuinfo") as f:
        cpu = next(
            linha.split(":", 1)[1].strip()
            for linha in f
            if linha.startswith("model name")
        )
except (OSError, StopIteration):
    pass

print("Ambiente: Google Colab (execução remota)")
print("Processador:", cpu)
print("Núcleos lógicos:", os.cpu_count())
print(f"RAM total disponibilizada: {ram_gb:.1f} GB")
print("Sistema operacional:", platform.platform())
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)

Ambiente: Google Colab (execução remota)
Processador: Intel(R) Xeon(R) CPU @ 2.20GHz
Núcleos lógicos: 2
RAM total disponibilizada: 12.7 GB
Sistema operacional: Linux-6.6.122+-x86_64-with-glibc2.39
Python: 3.13.16
scikit-learn: 1.6.1
NumPy: 2.1.3
Pandas: 2.2.3


In [ ]:
import importlib
import pandas as pd
import src.profiling as profiling

importlib.reload(profiling)

# Usar 4.000 registros de TREINAMENTO para medir inferência.
# O teste permanece reservado para a avaliação final.
indices_benchmark = X_train.sample(
    n=4000,
    random_state=42
).index

X_benchmark_se = X_train_se.loc[indices_benchmark]

# Medir 30 execuções de inferência
resultado = profiling.medir_inferencia(
    motor.predict,
    X_benchmark_se,
    n_rep=30,
    n_aquecimento=1,
    retornar_detalhes=True
)

# Salvar as medições individuais
arquivo_saida = (
    PROJETO / "logs/medicoes_inferencia_especialista.csv"
)

pd.DataFrame(resultado["detalhes"]).to_csv(
    arquivo_saida,
    index=False
)

# Estimar energia e emissões para UMA inferência do lote
energia = profiling.estimar_energia_carbono(
    resultado["tempo_medio_s"]
)

# Tamanho da base de regras e do motor em disco
arquivo_regras = PROJETO / "rules/regras_classificacao.txt"
arquivo_motor = PROJETO / "src/expert_system.py"

print("Repetições:", resultado["n_repeticoes"])
print("Amostras por repetição:", resultado["n_amostras"])
print("Número de regras:", len(base.regras))

print(f"\nTempo médio: {resultado['tempo_medio_s']:.6f} s")
print(f"Desvio padrão: {resultado['tempo_desvio_s']:.6f} s")
print(f"Tempo por amostra: {resultado['tempo_por_amostra_ms']:.6f} ms")
print(f"Vazão: {resultado['vazao_amostras_s']:.2f} amostras/s")

print(f"\nMaior pico de memória Python: {resultado['pico_memoria_kb']:.2f} KB")
print(f"Pico médio de memória Python: {resultado['pico_memoria_medio_kb']:.2f} KB")
print(f"Desvio dos picos: {resultado['pico_memoria_desvio_kb']:.2f} KB")

print(f"\nBase de regras: {arquivo_regras.stat().st_size / 1024:.2f} KB")
print(f"Motor Python: {arquivo_motor.stat().st_size / 1024:.2f} KB")

print(f"\nEnergia estimada: {energia['energia_kwh']:.10f} kWh")
print(f"Emissão estimada: {energia['emissao_co2_g']:.10f} g CO2")

print("\nMedições salvas em:", arquivo_saida)

Repetições: 30
Amostras por repetição: 4000
Número de regras: 11

Tempo médio: 0.025705 s
Desvio padrão: 0.001856 s
Tempo por amostra: 0.006426 ms
Vazão: 155614.77 amostras/s

Maior pico de memória Python: 318.91 KB
Pico médio de memória Python: 318.91 KB
Desvio dos picos: 0.01 KB

Base de regras: 1.81 KB
Motor Python: 7.36 KB

Energia estimada: 0.0000003213 kWh
Emissão estimada: 0.0000147158 g CO2

Medições salvas em: /content/drive/MyDrive/projeto-cardiovascular-ia-02/logs/medicoes_inferencia_especialista.csv


### 5.1 Tamanho do modelo serializado

O tamanho do arquivo de regras e o tamanho do código-fonte são apresentados acima. Para medir especificamente o **tamanho do modelo em disco**, serializamos temporariamente o motor de inferência, incluindo as regras, as estatísticas obtidas no treinamento e a estratégia de resposta padrão. O arquivo temporário é apagado depois da medição.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as pasta_temporaria:
    caminho_temporario = str(
        Path(pasta_temporaria) / "sistema_especialista.joblib"
    )
    tamanho_modelo_kb = profiling.obter_tamanho_modelo_disco(
        motor,
        caminho_temporario=caminho_temporario
    )

print(f"Tamanho do modelo serializado: {tamanho_modelo_kb:.2f} KB")

Tamanho do modelo serializado: 3.83 KB


## 6. Avaliação final no conjunto de teste

As regras, prioridades e a classe padrão devem estar congeladas. A execução desta etapa serve **somente para avaliação**, e não para escolher ou modificar regras. As métricas calculadas são Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC e Average Precision.


In [ ]:
import importlib
import numpy as np
import pandas as pd
import src.metrics as metricas

importlib.reload(metricas)

# Conferir o protocolo antes de avaliar
assert len(base.regras) == 11
assert len(X_test_se) == 4000
assert X_test_se.index.equals(y_test_c.index)
assert X_train_se.index.intersection(X_test_se.index).empty
assert not X_test_se.isna().any().any()

# Previsões do treinamento
pred_treino, esc_treino, _ = motor.inferir(X_train_se)

# Avaliação FINAL do conjunto de teste
pred_teste, esc_teste, regras_teste = motor.inferir(X_test_se)

# Conferir consistência das previsões
assert np.array_equal(
    pred_teste, motor.predict(X_test_se)
)

# Calcular métricas dos dois conjuntos
m_treino = metricas.calcular_metricas_classificacao(
    y_train_c, pred_treino, esc_treino
)

m_teste = metricas.calcular_metricas_classificacao(
    y_test_c, pred_teste, esc_teste
)

indicadores = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC",
    "Average_Precision"
]

comparacao = pd.DataFrame({
    "Treinamento": [m_treino[k] for k in indicadores],
    "Teste": [m_teste[k] for k in indicadores]
}, index=indicadores)

print("Desempenho do Sistema Especialista:")
display(comparacao.round(4))

# Matriz de confusão do teste
matriz = pd.DataFrame(
    m_teste["Confusion_Matrix"],
    index=["Real 0", "Real 1"],
    columns=["Previsto 0", "Previsto 1"]
)

print("\nMatriz de confusão — TESTE:")
display(matriz)

# Gap de generalização usando o F1
gap_f1 = (
    m_treino["F1-score"] - m_teste["F1-score"]
)

print(f"\nGap de F1 (treino - teste): {gap_f1:.4f}")

# Quantidade de registros sem regra no teste
print(
    "Registros com FALLBACK no teste:",
    int(np.sum(regras_teste == "FALLBACK"))
)

Desempenho do Sistema Especialista:


,Treinamento,Teste
Accuracy,0.7840,0.7728
Precision,0.6518,0.6331
Recall,0.6313,0.6119
F1-score,0.6414,0.6224
ROC-AUC,0.8086,0.7909
PR-AUC,0.6680,0.6597
Average_Precision,0.6314,0.6213



Matriz de confusão — TESTE:


,Previsto 0,Previsto 1
Real 0,2342,434
Real 1,475,749



Gap de F1 (treino - teste): 0.0191
Registros com FALLBACK no teste: 748


## 7. Interpretação dos resultados e limitações

No conjunto de teste, o Sistema Especialista obteve acurácia de **77,28%**, F1-score da classe positiva de **0,6224**, Recall de **0,6119**, ROC-AUC de **0,7909** e PR-AUC de **0,6597**.

Como aproximadamente 69,4% dos registros pertencem à classe negativa, um classificador que sempre previsse a classe 0 obteria essa mesma acurácia. O Sistema Especialista apresentou ganho de **7,88 pontos percentuais**, mas a acurácia isolada não é suficiente para caracterizar seu desempenho.

A matriz de confusão identificou **475 falsos negativos entre 1.224 registros positivos**. Isso representa aproximadamente 38,81% dos positivos não identificados, evidenciando uma limitação importante. Em contextos de triagem, a ocorrência de falsos negativos merece atenção especial.

O sistema utilizou `FALLBACK` em **748 dos 4.000 registros de teste (18,7%)**, demonstrando que parte dos registros ainda depende da classificação padrão.

As regras foram elaboradas com apoio de inteligência artificial e análises do treinamento. Seus escores correspondem às frequências de positivos entre os registros efetivamente classificados por cada regra ou pelo FALLBACK. Portanto, **não representam probabilidades clínicas calibradas**.

Como o motor apresenta até 12 escores distintos, correspondentes às 11 regras e ao FALLBACK, a curva Precision–Recall possui um número limitado de níveis. No teste, a PR-AUC trapezoidal foi de **0,6597**, enquanto a *Average Precision* foi de **0,6213**. Ambas são apresentadas separadamente por utilizarem métodos de cálculo diferentes.

O gap de F1 entre treinamento e teste foi de **0,0191**, sugerindo estabilidade relativa nessa divisão. Entretanto, uma única divisão estratificada não permite concluir definitivamente que o sistema apresenta baixa variância ou *underfitting*.

A comparação com árvores de decisão e ensembles será realizada nas próximas etapas. O Sistema Especialista constitui uma solução acadêmica interpretável, **não uma ferramenta de diagnóstico médico validada**, e suas regras não serão modificadas com base nos resultados do teste.

A classificação do sistema quanto ao ajuste (overfitting, underfitting ou bem ajustado) será consolidada no Notebook 07, considerando as diferenças entre treinamento e teste, o desempenho obtido e as limitações do modelo baseado em regras.